# 01 - Ingestão: CSV para Delta Lake (MinIO)

In [1]:
# ==============================================================================
# 1. Configuração e Conexão com o MinIO
# ==============================================================================
# Este bloco define as credenciais de acesso ao Object Storage (MinIO)
# que foi iniciado através do Docker Compose no ambiente local.

import os
from pyspark.sql import SparkSession

MINIO_ENDPOINT = "http://127.0.0.1:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "miniopass"
BUCKET_DESTINO = "datalake"

print(f"Endpoint configurado: {MINIO_ENDPOINT}")
print(f"Bucket de destino: {BUCKET_DESTINO}")

Endpoint configurado: http://127.0.0.1:9000
Bucket de destino: datalake


## 2. Inicialização da SparkSession com suporte a Delta e S3

In [2]:
# ==============================================================================
# 2. Inicialização da SparkSession com suporte a Delta e S3
# ==============================================================================
# Configuramos a sessão do Spark injetando os pacotes do Hadoop-AWS e 
# ativando as extensões do Delta Lake para permitir escrita em Object Storage.

spark = SparkSession.builder \
    .appName("Trabalho2-CSV-para-Delta") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark conectado ao MinIO com sucesso!")

26/10/13 20:33:48 WARN Utils: Your hostname, DESKTOP-53G9E1H resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/13 20:33:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/andrecittadin/.cache/pypoetry/virtualenvs/projeto-pesquisa-spark-parte2-CVx6iCAS-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/andrecittadin/.ivy2/cache
The jars for the packages stored in: /home/andrecittadin/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a8b624f0-f1c6-41e2-8a59-6e12908a35c1;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (247ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.

Spark conectado ao MinIO com sucesso!


## 3. Processo de Ingestão e Carga dos Dados

In [3]:
# ==============================================================================
# 3. Processo de Ingestão e Carga dos Dados (Corrigido para Inglês)
# ==============================================================================

# Agora com os nomes exatos dos ficheiros que o seu colega exportou
tabelas = ["students", "subjects", "classes", "grades", "teacher"]

for tabela in tabelas:
    # Caminho corrigido para a pasta correta: export/data/
    caminho_csv = f"export/data/{tabela}.csv"
    caminho_delta = f"s3a://{BUCKET_DESTINO}/bronze/{tabela}"
    
    print(f"A processar a tabela: {tabela}...")
    
    try:
        # Leitura do ficheiro CSV
        df = spark.read.csv(caminho_csv, header=True, inferSchema=True)
        
        # Escrita no Data Lake em formato Delta
        df.write.format("delta").mode("overwrite").save(caminho_delta)
        
        print(f" -> {tabela} guardada com sucesso no formato Delta!")
    except Exception as e:
        print(f" -> Erro ao processar a tabela {tabela}: {e}")

print("\nProcesso de Ingestão concluído!")

A processar a tabela: students...


26/10/13 20:33:57 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/10/13 20:34:02 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


 -> students guardada com sucesso no formato Delta!
A processar a tabela: subjects...
 -> subjects guardada com sucesso no formato Delta!
A processar a tabela: classes...
 -> classes guardada com sucesso no formato Delta!
A processar a tabela: grades...
 -> grades guardada com sucesso no formato Delta!
A processar a tabela: teacher...
 -> teacher guardada com sucesso no formato Delta!

Processo de Ingestão concluído!
